<a href="https://colab.research.google.com/github/41171135H/PL-Repo./blob/main/HW1_%E6%97%A5%E5%B8%B8%E6%94%AF%E5%87%BA%E9%80%9F%E7%AE%97%E8%88%87%E5%88%86%E6%94%A4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 日常支出速算與分攤（作業一）
*   目標：從 Sheet 讀「消費紀錄」→ 計總額/分類小計/AA 分攤 → 寫回 Sheet Summary 分頁。
*   AI 點子（可選）：請模型總結本週花錢習慣與建議（例如「外食過多」）。
*   Sheet 欄位：date, category, item, amount, payer
GoogleSheet:https://docs.google.com/spreadsheets/d/1noadDw8IUIle3wiei_-WwSHsWrA32Wh-thV25Y1T2eQ/edit?gid=0#gid=0

In [32]:
from google.colab import auth
auth.authenticate_user()
import pandas as pd
import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

In [33]:
gsheets = gc.open_by_url('https://docs.google.com/spreadsheets/d/1noadDw8IUIle3wiei_-WwSHsWrA32Wh-thV25Y1T2eQ/edit?gid=0#gid=0')

In [34]:
# 從 gsheets 載入 sheets
sheets = gsheets.worksheet('工作表1').get_all_values()
# 將 sheets1 資料載入 pd 的 DataFrame 進行分析
df = pd.DataFrame(sheets[1:], columns=sheets[0])
# 取得最前面的5筆資料
df.head()

,日期,時間,分類,品項,金額,付款人,地點,支付方式,備註
0,2026-03-07,17:58,晚餐,食物,109,我,0,現金,0
1,2026-03-02,14:59,藥,藥,398,我,0,現金,0
2,2026-02-28,20:20,晚餐,食物,274,我,0,現金,0
3,2026-03-02,12:29,加油,加油,82,我,0,現金,0
4,2026-03-05,18:30,晚餐,食物,178,我,0,現金,0


In [35]:
import datetime
# 讓使用者輸入資料
date_str = input("請輸入日期 (格式：YYYY-MM-DD): ")
# 檢查日期格式是否正確
datetime.datetime.strptime(date_str, '%Y-%m-%d')

time_str = input("請輸入時間 (格式：HH:MM): ")
# 檢查時間格式是否正確
datetime.datetime.strptime(time_str, '%H:%M')

item = input("請輸入品項: ")
amount = float(input("請輸入金額: "))

請輸入日期 (格式：YYYY-MM-DD): 2026-03-02
請輸入時間 (格式：HH:MM): 14:59
請輸入品項: 藥
請輸入金額: 398


In [38]:
# 創建一個新的 DataFrame 來代表你要新增的資料
new_data = pd.DataFrame([{
    '日期': date_str,
    '時間': time_str,
    '分類': '早餐',
    '品項': item,
    '金額': amount,
    '付款人': '我',
    '地點': '0',
    '支付方式': '現金',
    '備註': '0'
}])

# 使用 concat() 將新資料合併到舊的 df 中
df = pd.concat([df, new_data], ignore_index=True)

In [42]:
# 將 DataFrame 轉換成列表的列表 (list of lists)，這是 gspread 支援的資料格式
data_to_write = df.values.tolist()

# 第一步：取得工作表物件
# gsheets.worksheet('記帳本') 返回的是工作表物件，而非其內容
worksheet = gsheets.worksheet('工作表1')

In [43]:
data_to_write[0]

['2026-03-07', '17:58', '晚餐', '食物', '109', '我', '0', '現金', '0']

In [44]:
# 第二步：將資料寫入工作表
# 將 nan 值替換為 0
row_to_append = [item if pd.notna(item) else 0 for item in data_to_write[-1]]
worksheet.append_rows(values=[row_to_append], value_input_option='USER_ENTERED')
print("資料已成功寫入 Google 工作表！")

資料已成功寫入 Google 工作表！
